# Noisy Expected Improvement (NEI) with QMC

Demonstrates how quasi-Monte Carlo sampling improves the estimation of
Expected Improvement (EI) in Bayesian Optimization, using a simple 1D
example with a manual Gaussian process.

Based on the Python QMCPy demo `nei_demo.ipynb`.

*Note: The Python version uses scipy/sklearn for GP fitting. This Julia
version constructs the GP manually using Cholesky decomposition, keeping
the focus on how QMC improves EI estimation.*

In [ ]:
using QMC
using LinearAlgebra
using Statistics
using Printf

## Test Function

A 1D function with multiple local maxima — a typical surrogate optimization target.

In [ ]:
# Target function
yf(x) = cos(10x) * exp(0.2x) + exp(-5(x - 0.4)^2)

# Observed data with noise variances
x_obs = [0.1, 0.2, 0.4, 0.7, 0.9]
y_obs = yf.(x_obs)
v_obs = [0.001, 0.05, 0.01, 0.1, 0.4]  # noise variances

println("Observed data:")
for i in eachindex(x_obs)
    println("  x=\$(x_obs[i]), y=\$(round(y_obs[i], digits=4)), noise_var=\$(v_obs[i])")
end
println("\nCurrent best: y_max = \$(round(maximum(y_obs), digits=4)) at x = \$(x_obs[argmax(y_obs)])")

## Gaussian Process Construction

Build a GP posterior using a squared-exponential (Gaussian) kernel with
Cholesky factorization.

In [ ]:
# Gaussian kernel
function gp_kernel(x, z; shape=4.1, pv=0.9)
    return pv * exp(-shape^2 * (x - z)^2)
end

function gp_kernel_matrix(xs, zs; shape=4.1, pv=0.9)
    K = [gp_kernel(x, z; shape=shape, pv=pv) for x in xs, z in zs]
    return K
end

# Build GP posterior
K_dd = gp_kernel_matrix(x_obs, x_obs) + diagm(v_obs)  # add noise
L_dd = cholesky(Symmetric(K_dd + 1e-10I)).L

# Prediction function
x_pred = range(0, 1, length=200)
K_pd = gp_kernel_matrix(collect(x_pred), x_obs)
K_pp = gp_kernel_matrix(collect(x_pred), collect(x_pred))

alpha = L_dd' \ (L_dd \ y_obs)
mu_pred = K_pd * alpha

V = L_dd \ K_pd'
cov_pred = K_pp - V' * V
sigma_pred = sqrt.(max.(diag(cov_pred), 0.0))

println("GP posterior computed over \$(length(x_pred)) prediction points")
println("Posterior mean range: [\$(round(minimum(mu_pred), digits=3)), \$(round(maximum(mu_pred), digits=3))]")
println("Max posterior std: \$(round(maximum(sigma_pred), digits=4))")

## Expected Improvement

Estimate EI at each prediction point using Monte Carlo draws from the GP posterior,
comparing IID vs QMC sampling.

In [ ]:
using Distributions: Normal, cdf, pdf, quantile

y_best = maximum(y_obs)

# Analytical EI (for comparison)
function analytical_ei(mu, sigma, y_best)
    if sigma < 1e-10
        return max(mu - y_best, 0.0)
    end
    z = (mu - y_best) / sigma
    d = Normal(0, 1)
    return sigma * (z * cdf(d, z) + pdf(d, z))
end

ei_analytical = [analytical_ei(mu_pred[i], sigma_pred[i], y_best) for i in eachindex(x_pred)]

println("Analytical EI:")
println("  Max EI = \$(round(maximum(ei_analytical), digits=6)) at x = \$(round(x_pred[argmax(ei_analytical)], digits=3))")

In [ ]:
# Monte Carlo EI estimation
function mc_ei(mu_pred, sigma_pred, y_best, n_samples; use_qmc=false, seed=7)
    n_pred = length(mu_pred)
    ei = zeros(n_pred)
    
    for i in 1:n_pred
        if sigma_pred[i] < 1e-10
            ei[i] = max(mu_pred[i] - y_best, 0.0)
            continue
        end
        # Generate samples
        if use_qmc
            dd = DigitalNetB2(1; seed=seed)
            u = gen_samples(dd, n_samples)[:, 1]
        else
            dd = IIDStdUniform(1; seed=seed)
            u = gen_samples(dd, n_samples)[:, 1]
        end
        # Transform to normal
        d = Normal(0, 1)
        z = [quantile(d, max(min(ui, 1-1e-10), 1e-10)) for ui in u]
        # GP draws
        draws = mu_pred[i] .+ sigma_pred[i] .* z
        # EI = E[max(f - y_best, 0)]
        improvements = max.(draws .- y_best, 0.0)
        ei[i] = mean(improvements)
    end
    return ei
end

# Compare at different sample sizes
println("\nEI estimation comparison (max EI location):\n")
println("Samples    IID location   QMC location   Analytical: \$(round(x_pred[argmax(ei_analytical)], digits=3))")
println("-"^65)

for n_samples in [16, 64, 256, 1024]
    ei_iid = mc_ei(mu_pred, sigma_pred, y_best, n_samples; use_qmc=false)
    ei_qmc = mc_ei(mu_pred, sigma_pred, y_best, n_samples; use_qmc=true)
    
    loc_iid = x_pred[argmax(ei_iid)]
    loc_qmc = x_pred[argmax(ei_qmc)]
    
    @printf("%-10d %-14.3f %-14.3f\n", n_samples, loc_iid, loc_qmc)
end

## EI Error Analysis

Measure how quickly IID and QMC converge to the analytical EI.

In [ ]:
# RMSE of EI estimate vs analytical
println("RMSE of EI estimate vs analytical:\n")
println("Samples    IID RMSE       QMC RMSE       Ratio")
println("-"^55)

for n_samples in [8, 16, 32, 64, 128, 256, 512, 1024]
    ei_iid = mc_ei(mu_pred, sigma_pred, y_best, n_samples; use_qmc=false)
    ei_qmc = mc_ei(mu_pred, sigma_pred, y_best, n_samples; use_qmc=true)
    
    rmse_iid = sqrt(mean((ei_iid .- ei_analytical).^2))
    rmse_qmc = sqrt(mean((ei_qmc .- ei_analytical).^2))
    ratio = rmse_iid / max(rmse_qmc, 1e-15)
    
    @printf("%-10d %-14.2e %-14.2e %.1f×\n", n_samples, rmse_iid, rmse_qmc, ratio)
end

println("\nQMC consistently achieves lower RMSE for the same sample budget,")
println("demonstrating the benefit of low-discrepancy sampling for EI estimation.")